In [13]:
# Se cargan las bibliotecas de pandas, numpy y matplotlib
import pandas as pd 
from pandas import ExcelWriter
from pandas import ExcelFile
import numpy as np
import matplotlib. pyplot as plt
from scipy import stats
from scipy.stats import anderson, normaltest, probplot, t
import time

print("Bibliotecas importadas correctamente")

Bibliotecas importadas correctamente


In [14]:
#Lectura de los datos de excel 
datos = pd.read_excel('Dataset.xlsx')
BPM = datos ['BPM']

In [15]:
#pasa los datos de BPM a un array de 8 bits
def ints_to_binary_array(integers):
    binary_array = [bin(num)[2:].zfill(8) for num in integers]
    return binary_array

BPM = round(BPM)
n = len(BPM)
mensaje_7_4 = []
mensaje_15_11 = []
BPM = BPM.astype(int) #hace que todos los datos sean de tipo int
binario = ints_to_binary_array(BPM)

In [16]:
#Codificacion Hamming (7,4)

promedio = 0
for _ in range(200):
    start = time.time()
    for j in range(n):
        # Se toman los primeros 4 bits, primer nibble
        d = binario[j][0:4]
        d0 = int(d[0])
        d1 = int(d[1])
        d2 = int(d[2])
        d3 = int(d[3])

        #XOR's de hamming
        p1 = d0 ^ d1 ^ d3
        p2 = d0 ^ d2 ^ d3
        p3 = d1 ^ d2 ^ d3

        codeword1 = [p1, p2, d0, p3, d1, d2, d3] # p => paridad    d => datos
        mensaje_7_4.append(codeword1)

        # Se toman los ultimos 4 bits, segundo nibble
        d = binario[j][4:8]
        d0 = int(d[0])
        d1 = int(d[1])
        d2 = int(d[2])
        d3 = int(d[3])

        p1 = d0 ^ d1 ^ d3
        p2 = d0 ^ d2 ^ d3
        p3 = d1 ^ d2 ^ d3

        codeword2 = [p1, p2, d0, p3, d1, d2, d3]
        mensaje_7_4.append(codeword2)

    end = time.time()
    promedio += (end - start)

promedio /= 200
print("Tiempo promedio para Hamming (7,4) =   ", promedio)


Tiempo promedio para Hamming (7,4) =    0.0003185415267944336


In [17]:
#Codificacion Hamming (15,11)
 
promedio = 0
for _ in range(200):
    start = time.time()
    
    for j in range(n):
        d = binario[j]  
        d0 = int(d[0])
        d1 = int(d[1])
        d2 = int(d[2])
        d3 = int(d[3])
        d4 = int(d[4])
        d5 = int(d[5])
        d6 = int(d[6])
        d7 = int(d[7])
        
        # Los bits restantes se rellenan con ceros para llegar a 11 bits
        d8  = 0
        d9  = 0
        d10 = 0

        #XOR's de hamming para calcular las paridades
        # p1 cubre las posiciones cuyo LSB es 1 → 1,3,5,7,9,11,13,15
        p1 = d0 ^ d1 ^ d2 ^ d3 ^ d4 ^ d5 ^ d6 ^ d7 ^ d8 ^ d9 ^ d10

        # p2 cubre posiciones donde el segundo bit del índice es 1 → 2,3,6,7,10,11,14,15
        p2 = d0 ^ d1 ^ d3 ^ d4 ^ d6 ^ d7 ^ d9 ^ d10

        # p4 cubre posiciones cuyo tercer bit es 1 → 4,5,6,7,12,13,14,15
        p4 = d0 ^ d2 ^ d3 ^ d5 ^ d7 ^ d8 ^ d9 ^ d10
        
        # p8 cubre posiciones cuyo cuarto bit es 1 → 8,9,10,11,12,13,14,15
        p8 = d0 ^ d1 ^ d2 ^ d3 ^ d4 ^ d5 ^ d6 ^ d7 

        # Se construye la palabra Hamming (15,11)
        # posiciones 1..15
        codeword = [p1, p2, d0, p4, d1, d2,d3,p8, d4, d5, d6, d7, d8, d9, d10]

        mensaje_15_11.append(codeword)

    end = time.time()
    promedio += (end - start)

promedio /= 200
print("Tiempo promedio para Hamming (15,11) = ", promedio)

Tiempo promedio para Hamming (15,11) =  0.00029382705688476565


In [18]:
file_path = "Datos_codificados/Codificacion_hamming_7_4.txt"

with open(file_path, 'w') as file:
    for i in range(0,len(mensaje_7_4)):
        file.write(str(mensaje_7_4[i]))
        file.write(",")


file_path = "Datos_codificados/Codificacion_hamming_15_11.txt"

with open(file_path, 'w') as file:
    for i in range(0,n):
        file.write(str(mensaje_15_11[i]))
        file.write(",")

    